In [85]:
import requests
import pandas as pd
import numpy as np
import datetime as dt

sales_url = "https://datacatalog.cookcountyil.gov/resource/wvhk-k5uv.json"
proximity_url = "https://datacatalog.cookcountyil.gov/resource/ydue-e5u3.json"

sales_r = requests.get(sales_url, params = {
    "$where": "year = 2023 AND sale_filter_less_than_10k = false AND sale_filter_deed_type = false "
    "AND sale_filter_same_sale_within_365 = false AND is_multisale = false",
    '$select': "pin,year,sale_date,sale_price,class,township_code,nbhd,deed_type,num_parcels_sale",
    "$order": "row_id"
})

sales_r.raise_for_status()
sales_rows = sales_r.json()
sales = pd.DataFrame(sales_rows)

sales["pin"] = sales["pin"].str.zfill(14)
sales["pin10"] = sales["pin"].str[:10]
sales["sale_price"] = pd.to_numeric(sales["sale_price"], errors="coerce")
sales["sale_date"] = pd.to_datetime(sales["sale_date"])

pin_list = sales['pin10'].unique().tolist()
pin_chunks = [pin_list[i:i+200] for i in range(0, len(pin_list), 200)]

proximity = pd.concat(
    [pd.DataFrame(requests.get(proximity_url, params = {
    "$where": "year = 2023 AND pin10 IN (" + ",".join(f"'{p}'" for p in ch) + ")",
    "$select": "pin10,year,nearest_cta_stop_dist_ft,nearest_metra_stop_dist_ft,num_bus_stop_in_half_mile,"
    "nearest_major_road_dist_ft,nearest_park_dist_ft,lake_michigan_dist_ft",
    "$order": "pin10"
}).json()) for ch in pin_chunks], ignore_index = True)


display(proximity, sales)

,pin10,year,nearest_cta_stop_dist_ft,nearest_metra_stop_dist_ft,num_bus_stop_in_half_mile,nearest_major_road_dist_ft,nearest_park_dist_ft,lake_michigan_dist_ft
0,0925427042,2023,11108.72875273877,3911.781434009471,28,371.05891167642653,1117.6416558475225,39827.679268198386
1,0936419107,2023,6397.59090947421,2033.5409823492423,39,1088.053480366604,952.6061082532915,40042.19807033175
2,0936422005,2023,6137.984459782503,1583.8374748108815,45,2011.9219671169872,110.61627958988993,41044.74834348826
3,0936426061,2023,5972.228355402146,1690.9617742380442,45,2644.3122565979543,109.7993985999925,41754.49739731001
4,1031408025,2023,7968.779139332947,3632.796878005821,17,450.14974338424076,3975.587495445511,35258.67019226818
...,...,...,...,...,...,...,...,...
958,3307417034,2023,71613.21146271877,35251.449311332726,0,3911.7625190607964,750.1148702309993,64838.091942195126
959,3307421003,2023,72101.01802495391,36213.61388208982,0,4871.974332044205,1570.4688031897383,64962.023590239856
960,3317305057,2023,79255.15653794572,40979.55365977999,0,1953.861408046219,2244.8765135619296,71581.78900439957
961,3318202004,2023,76193.70236101425,37654.06695421883,0,2042.1056589234572,2319.6423678267047,69332.62272388696


,pin,year,sale_date,sale_price,class,township_code,nbhd,deed_type,num_parcels_sale,pin10
0,20223260160000,2023,2023-01-12,1000000,318,70,70030,Other,1,2022326016
1,25151170160000,2023,2023-01-18,160000,203,70,70220,Warranty,1,2515117016
2,20023020390000,2023,2023-01-19,40000,205,70,70010,Warranty,1,2002302039
3,20114000191005,2023,2023-01-10,244000,299,70,70020,Warranty,1,2011400019
4,20352090130000,2023,2023-01-25,55000,203,70,70080,Warranty,1,2035209013
...,...,...,...,...,...,...,...,...,...,...
995,16041040270000,2023,2023-01-26,330000,211,77,77011,Warranty,1,1604104027
996,14311060501001,2023,2023-01-18,312500,299,77,77170,Warranty,1,1431106050
997,17064090461002,2023,2023-01-18,540000,299,77,77051,Warranty,1,1706409046
998,16133100090000,2023,2023-01-13,270000,211,77,77102,Warranty,1,1613310009


In [86]:
merged = sales.merge(proximity, on = ['pin10', 'year'], how = 'left', indicator = True)
merged['_merge'].value_counts()

_merge
both          1000
left_only        0
right_only       0
Name: count, dtype: int64

In [89]:
m = merged.dropna(subset=["sale_price", "nearest_cta_stop_dist_ft", "nearest_metra_stop_dist_ft"]).copy()
m.to_csv('./data/merged_data.csv')